In [ ]:
!pip install ultralytics -q

from google.colab import drive
drive.mount('/content/drive')

print("Ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.4 MB/s eta 0:00:00
Mounted at /content/drive
Ready


In [ ]:
import os
import cv2
from pathlib import Path

crop_dir = '/content/crops'
for cat in ['STATIONARY', 'LYING_DOWN', 'OBSCURED']:
    os.makedirs(f'{crop_dir}/{cat}', exist_ok=True)

img_dir = '/content/drive/MyDrive/ARIA/data/combined/train/images'
label_dir = '/content/drive/MyDrive/ARIA/data/combined/train/labels'

label_files = list(Path(label_dir).glob('*.txt'))
print(f"Found {len(label_files)} label files")

count = 0
for idx, label_file in enumerate(label_files):
    if idx % 500 == 0:
        print(f"Processing {idx}/{len(label_files)}... ({count} crops)")

    img_name = label_file.stem
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = os.path.join(img_dir, img_name + ext)
        if os.path.exists(candidate):
            img_path = candidate
            break
    if img_path is None:
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    with open(label_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = max(0, int((cx - bw/2) * w))
            y1 = max(0, int((cy - bh/2) * h))
            x2 = min(w, int((cx + bw/2) * w))
            y2 = min(h, int((cy + bh/2) * h))

            crop = img[y1:y2, x1:x2]
            if crop.shape[0] < 10 or crop.shape[1] < 10:
                continue

            aspect = crop.shape[0] / crop.shape[1]
            if aspect > 1.8:
                category = 'STATIONARY'
            elif aspect < 0.6:
                category = 'LYING_DOWN'
            else:
                category = 'OBSCURED'

            resized = cv2.resize(crop, (224, 224))
            cv2.imwrite(f'{crop_dir}/{category}/crop_{count}.jpg', resized)
            count += 1

print(f"\nDone! {count} crops")
for cat in ['STATIONARY', 'LYING_DOWN', 'OBSCURED']:
    n = len([f for f in os.listdir(f'{crop_dir}/{cat}') if f.endswith('.jpg')])
    print(f"  {cat}: {n}")

Found 5123 label files
Processing 0/5123... (0 crops)
Processing 500/5123... (680 crops)
Processing 1000/5123... (1406 crops)
Processing 1500/5123... (1965 crops)
Processing 2000/5123... (2517 crops)
Processing 2500/5123... (3158 crops)
Processing 3000/5123... (3822 crops)
Processing 3500/5123... (4373 crops)
Processing 4000/5123... (5043 crops)
Processing 4500/5123... (5456 crops)
Processing 5000/5123... (5900 crops)

Done! 6024 crops
  STATIONARY: 3466
  LYING_DOWN: 194
  OBSCURED: 2364


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os

crop_dir = '/content/crops'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(crop_dir, transform=transform)
print(f"Classes: {dataset.classes}")
print(f"Total samples: {len(dataset)}")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)

model = models.efficientnet_b0(weights='IMAGENET1K_V1')
num_classes = len(dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.cuda()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    print(f"Epoch {epoch+1}/10 | Loss: {running_loss/len(train_loader):.4f} | Train: {100*correct/total:.1f}% | Val: {100*val_correct/val_total:.1f}%")

save_path = '/content/drive/MyDrive/ARIA/models/efficientnet_distress.pt'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': dataset.classes,
    'num_classes': num_classes
}, save_path)
print(f"\nSaved to {save_path}")
print(f"Classes: {dataset.classes}")

Classes: ['LYING_DOWN', 'OBSCURED', 'STATIONARY']
Total samples: 6024
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 120MB/s]


Epoch 1/10 | Loss: 0.3151 | Train: 87.9% | Val: 91.8%
Epoch 2/10 | Loss: 0.2228 | Train: 90.7% | Val: 93.0%
Epoch 3/10 | Loss: 0.2050 | Train: 91.9% | Val: 88.6%
Epoch 4/10 | Loss: 0.2006 | Train: 92.3% | Val: 91.4%
Epoch 5/10 | Loss: 0.1573 | Train: 93.7% | Val: 91.4%
Epoch 6/10 | Loss: 0.1598 | Train: 94.3% | Val: 91.3%
Epoch 7/10 | Loss: 0.1504 | Train: 94.6% | Val: 92.0%
Epoch 8/10 | Loss: 0.1151 | Train: 95.5% | Val: 92.9%
Epoch 9/10 | Loss: 0.0945 | Train: 96.4% | Val: 92.0%
Epoch 10/10 | Loss: 0.1125 | Train: 95.8% | Val: 92.0%

Saved to /content/drive/MyDrive/ARIA/models/efficientnet_distress.pt
Classes: ['LYING_DOWN', 'OBSCURED', 'STATIONARY']
